# Whispers — Phase 1 GRPO training (Kaggle 2× T4 edition)

This notebook runs end-to-end on a free **Kaggle notebook with 2× NVIDIA T4 GPUs**. It:

1. Clones the Whispers OpenEnv repo into `/kaggle/working/whispers`.
2. Installs the vanilla HF + bitsandbytes + PEFT + TRL stack (no Unsloth — Unsloth is single-GPU only) plus **[Liger-Kernel](https://github.com/linkedin/Liger-Kernel)** for Triton-fused attention/MLP kernels.
3. Loads `Qwen/Qwen2.5-3B-Instruct` in 4-bit and **shards it across both T4s via `device_map='auto'`**, then patches it with Liger-Kernel for ~30–60% faster forward/backward and ~60% lower activation memory.
4. Drives a GRPO-style rollout loop against the in-process `WhispersEnv` (six tasks, curriculum-weighted).
5. Saves three plots into `/kaggle/working/whispers/assets/`.

**Theme**: Multi-Agent Interactions (cooperation, competition, negotiation, coalition formation).

---

### Why no Unsloth on Kaggle?
Unsloth's optimised kernels target a single GPU. To use both T4s we use the standard HuggingFace stack with `device_map='auto'`, which auto-shards the model across both devices — and we recover most of Unsloth's speedup by layering **Liger-Kernel** (Triton-fused RoPE / RMSNorm / SwiGLU / fused-CE) on top, since Liger composes cleanly with multi-GPU sharding.

### Why Qwen 3B instead of 1.5B (Colab default)?
Two T4s give ~32 GB of VRAM. A 3B model (~2 GB at 4-bit) leaves plenty of headroom for activations on the second GPU. Bump `WHISPERS_MODEL=Qwen/Qwen2.5-7B-Instruct` if you want to use most of the VRAM.

### Runtime budget
With the defaults below (`GRPO_STEPS=80`, `GENERATIONS_PER_STEP=4`, `MAX_STEPS_PER_EP=12`), the rollout loop completes in **~2 hours** on Kaggle's 2× T4 setup, well inside the 9-hour notebook quota.

---

> **One-time Kaggle setup:** in the right-hand sidebar, **Settings → Accelerator → GPU T4 ×2**, **Settings → Internet → On**, and (optionally) **Add-ons → Secrets** to add `WANDB_API_KEY` / `HF_TOKEN`.

> All long-running cells are wrapped in `try/except` so a re-run in CPU-only mode still produces synthetic curves for the headline plots.

## 0. Environment setup

In [ ]:
import os, sys, subprocess

IN_KAGGLE = bool(os.environ.get('KAGGLE_KERNEL_RUN_TYPE') or os.path.exists('/kaggle/working'))
IN_COLAB = 'google.colab' in sys.modules
PLATFORM = 'kaggle' if IN_KAGGLE else 'colab' if IN_COLAB else 'local'

REPO_URL = os.environ.get('WHISPERS_REPO', 'https://huggingface.co/spaces/varn03/whispers')
if IN_KAGGLE:
    REPO_DIR = '/kaggle/working/whispers'
elif IN_COLAB:
    REPO_DIR = '/content/whispers'
else:
    REPO_DIR = '..'

if PLATFORM in ('kaggle', 'colab') and not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)

sys.path.insert(0, REPO_DIR)
print(f'platform={PLATFORM}  repo={REPO_DIR}')

In [ ]:
%%capture
# Kaggle ships with a recent-ish HF stack but we re-install the few packages
# we need at known-good versions.
#
# Notable choices:
#   * NO Unsloth — it is single-GPU only.
#   * Liger-Kernel for Triton-fused RoPE / RMSNorm / SwiGLU / fused-CE.
#     Liger composes with device_map='auto' + bnb 4-bit + PEFT, giving
#     Unsloth-style ~30-60% speedup AND multi-GPU support at the same time.
if PLATFORM in ('kaggle', 'colab'):
    !pip install -q -U pip
    !pip install -q 'transformers>=4.45,<4.50' 'accelerate>=1.0' 'bitsandbytes>=0.44'
    !pip install -q 'peft>=0.13' 'trl<0.13.0'
    # Liger-Kernel version pinning is finicky:
    #   * 0.6+ requires transformers>=4.49
    #   * 0.7+ requires transformers>=4.52
    #   * 0.5.9 / 0.5.10 added a `lce_forward_deprecated` symbol that
    #     monkey_patch.py imports but which clashes with transformers 4.46.x
    # We're pinned to transformers<4.50 (TRL <0.13 compat), so we hold Liger
    # at <=0.5.8 which is the last clean release for our transformers range.
    # If even this fails (e.g. you swapped the model family), the model load
    # cell falls back to vanilla HF kernels with no error.
    !pip install -q --upgrade --force-reinstall 'liger-kernel>=0.5.4,<=0.5.8'
    !pip install -q wandb 'pydantic>=2.9' python-dotenv
else:
    print('Local mode — assumes deps already installed')

In [ ]:
# Safety net: if pip just upgraded a native lib already loaded in the kernel
# (most commonly numpy or torch), the in-memory ABI no longer matches the new
# .so files and the next heavy import crashes. We detect the mismatch and
# auto-restart; pip caches mean the second pass is fast.
#
# We also check liger_kernel because we may have just downgraded it (e.g.
# replacing a 0.7.x install from a previous run with our pinned 0.5.x), in
# which case any cached liger module bytes in sys.modules will be stale.
if PLATFORM in ('kaggle', 'colab'):
    import importlib.metadata as _im
    import sys as _sys
    _restart = False

    import numpy as _np
    _disk = _im.version('numpy')
    if _disk != _np.__version__:
        print(f'numpy in kernel ({_np.__version__}) != on disk ({_disk}).')
        _restart = True
    else:
        print(f'numpy OK at {_np.__version__}')

    if 'liger_kernel' in _sys.modules:
        try:
            _lk_disk = _im.version('liger-kernel')
            _lk_mod = _sys.modules['liger_kernel']
            _lk_mem = getattr(_lk_mod, '__version__', None)
            if _lk_mem and _lk_mem != _lk_disk:
                print(f'liger_kernel in kernel ({_lk_mem}) != on disk ({_lk_disk}).')
                _restart = True
        except Exception:
            # if liger is partially imported from a previous failed run,
            # drop it from sys.modules so the next import is clean
            for _k in [k for k in _sys.modules if k.startswith('liger_kernel')]:
                del _sys.modules[_k]
            print('cleared stale liger_kernel modules from sys.modules')

    if _restart:
        print('Restarting runtime so the new ABI is picked up...')
        os.kill(os.getpid(), 9)
    else:
        print('all good — no restart needed.')

In [ ]:
# Credentials. Order of preference:
#   1. anything already exported in the kernel,
#   2. Kaggle Secrets (sidebar Add-ons → Secrets) when running on Kaggle,
#   3. Colab Secrets (sidebar key icon) when running on Colab,
#   4. .env file in the repo root.
# Nothing is required — WANDB_API_KEY and HF_TOKEN are both optional. WANDB
# only enables cloud logging; HF_TOKEN is only needed for gated model pulls.
import os

def _maybe_set(key: str, value):
    if value and not os.environ.get(key):
        os.environ[key] = str(value)

if PLATFORM == 'kaggle':
    try:
        from kaggle_secrets import UserSecretsClient
        _ks = UserSecretsClient()
        for k in ('WANDB_API_KEY', 'HF_TOKEN', 'HUGGINGFACE_HUB_TOKEN'):
            try:
                _maybe_set(k, _ks.get_secret(k))
            except Exception:
                pass  # secret not added or not enabled for this notebook
    except ImportError:
        pass

if PLATFORM == 'colab':
    try:
        from google.colab import userdata
        for k in ('WANDB_API_KEY', 'HF_TOKEN', 'HUGGINGFACE_HUB_TOKEN'):
            try:
                _maybe_set(k, userdata.get(k))
            except Exception:
                pass
    except ImportError:
        pass

try:
    from dotenv import load_dotenv
    _candidate = os.path.join(REPO_DIR, '.env')
    if os.path.isfile(_candidate):
        load_dotenv(_candidate, override=False)
        print(f'Loaded credentials from {_candidate}')
except ImportError:
    pass

print('WANDB_API_KEY set?', bool(os.environ.get('WANDB_API_KEY')))
print('HF_TOKEN set?    ', bool(os.environ.get('HF_TOKEN') or os.environ.get('HUGGINGFACE_HUB_TOKEN')))

In [ ]:
import os, json, math, random, time, subprocess, logging
from collections import defaultdict
from pathlib import Path
import numpy as np
import torch
import matplotlib.pyplot as plt

# Quiet the Whispers env logger. Malformed-action `ToolError`s are an *expected*
# part of training (the agent is already penalised via shaping and the error
# is surfaced via info["tool_error"]) but they spam Kaggle's stderr.
logging.getLogger('whispers').setLevel(logging.ERROR)
logging.getLogger('whispers.env').setLevel(logging.ERROR)

ASSETS = Path(REPO_DIR) / 'assets'
ASSETS.mkdir(parents=True, exist_ok=True)
print('assets ->', ASSETS.resolve())

# GPU census so the user can see both T4s came up.
print('CUDA available :', torch.cuda.is_available())
print('GPU count      :', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'  GPU {i}: {p.name}  {p.total_memory/1024**3:.1f} GB')

## 1. Smoke-test the environment
We confirm the `WhispersEnv` reset/step/grade contract before plugging it into GRPO.

In [ ]:
from whispers.env import WhispersEnv
from whispers.models import WhispersAction

env = WhispersEnv(task_id='t1', seed=0)
obs = env.reset()
print('role=', obs.role, 'inbox=', len(obs.inbox), 'legal_tools=', obs.legal_tools)
obs, r, done, info = env.step(WhispersAction(tool='wait'))
print('step1 reward=', round(r.value, 3), 'done=', done)
report = {'location': {'value': 'Reactor 7', 'confidence': 0.5},
          'incident': {'value': 'fire alarm', 'confidence': 0.5},
          'time': {'value': '03:14', 'confidence': 0.5},
          'casualties': {'value': '0', 'confidence': 0.5}}
obs, r, done, info = env.step(WhispersAction(tool='publish', final_report=report))
print('terminal reward.value=', round(r.value, 3), 'episode_score=', round(info.get('episode_score', 0.0), 3))

## 2. Load Qwen 2.5-3B-Instruct (4-bit) sharded across both T4s, accelerated with Liger-Kernel

We use the vanilla HuggingFace stack (`transformers + bitsandbytes + peft`) rather than Unsloth so `device_map='auto'` can shard the model across both GPUs. With 2× 16 GB T4 (~32 GB total) you can freely bump to `Qwen/Qwen2.5-7B-Instruct` if you want to use more of the VRAM.

We then patch the model with **[Liger-Kernel](https://github.com/linkedin/Liger-Kernel)**, a set of Triton-fused kernels for RoPE / RMSNorm / SwiGLU / fused cross-entropy. Liger gives us Unsloth-style ~30–60% speedup and ~60% lower activation memory **while remaining multi-GPU compatible** — the missing piece we lose by dropping Unsloth.

In [ ]:
MODEL_NAME = os.environ.get('WHISPERS_MODEL', 'Qwen/Qwen2.5-3B-Instruct')
MAX_SEQ_LEN = 2048
LORA_RANK = 16

model = tokenizer = None
LIGER_APPLIED = False
try:
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

    # --- Liger-Kernel patching --------------------------------------------------
    # Apply Triton-fused RoPE/RMSNorm/SwiGLU/fused-CE kernels for the chosen
    # model family BEFORE we instantiate the model. This composes with bnb 4-bit
    # quant and PEFT LoRA without any further changes to the rollout loop.
    # We pick the patcher by model family so swapping MODEL_NAME just works.
    def _apply_liger_for(model_id: str):
        """Best-effort Liger patch. Returns (applied, reason). NEVER raises:
        any Liger version-mismatch or internal import error must NOT block
        the model load. We catch broad Exception at every step because
        liger_kernel.transformers uses lazy __getattr__ -> monkey_patch.py
        which can re-raise ImportError on attribute access (e.g. a broken
        sibling submodule like gemma.py with a missing symbol)."""
        name = model_id.lower()
        try:
            import liger_kernel.transformers as _lk
        except Exception as _e:
            return False, f'liger import failed: {type(_e).__name__}: {_e}'
        candidates = (
            ('qwen2',   'apply_liger_kernel_to_qwen2'),
            ('qwen',    'apply_liger_kernel_to_qwen2'),
            ('llama',   'apply_liger_kernel_to_llama'),
            ('mistral', 'apply_liger_kernel_to_mistral'),
            ('gemma2',  'apply_liger_kernel_to_gemma2'),
            ('gemma',   'apply_liger_kernel_to_gemma'),
            ('phi3',    'apply_liger_kernel_to_phi3'),
        )
        for substr, patcher_name in candidates:
            if substr not in name:
                continue
            # NOTE: do NOT use hasattr(_lk, patcher_name). hasattr only
            # suppresses AttributeError, but Liger's lazy __getattr__ can
            # raise ImportError when a sibling submodule is broken.
            try:
                patcher = getattr(_lk, patcher_name, None)
            except Exception as _e:
                return False, f'getattr({patcher_name}) raised {type(_e).__name__}: {_e}'
            if patcher is None:
                continue
            try:
                patcher(
                    rope=True, rms_norm=True, swiglu=True,
                    cross_entropy=False,            # leave HF CE for PEFT compat
                    fused_linear_cross_entropy=True,
                )
                return True, f'patched via {patcher_name}'
            except Exception as _e:
                return False, f'{patcher_name}() raised {type(_e).__name__}: {_e}'
        return False, f'no liger patcher matched model id "{model_id}"'

    # Outer guard: even if _apply_liger_for itself somehow raises (it shouldn't),
    # we want LIGER_APPLIED=False rather than aborting the whole model load.
    try:
        LIGER_APPLIED, _liger_reason = _apply_liger_for(MODEL_NAME)
    except Exception as _e:
        LIGER_APPLIED, _liger_reason = False, f'unhandled: {type(_e).__name__}: {_e}'
    if LIGER_APPLIED:
        print(f'Liger-Kernel: ON   ({_liger_reason})')
    else:
        print(f'Liger-Kernel: OFF  ({_liger_reason}) — falling back to vanilla HF kernels')
    # ---------------------------------------------------------------------------

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
    )
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    # device_map='auto' lets accelerate split layers across the two T4s.
    # max_memory caps each GPU so accelerate is FORCED to spread the load
    # rather than dumping everything on GPU 0 just because it fits.
    n_gpus = max(1, torch.cuda.device_count())
    max_memory = {i: '12GiB' for i in range(n_gpus)}
    if n_gpus < 2:
        print('Warning: only 1 GPU detected. Enable "GPU T4 ×2" in Kaggle settings.')

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map='auto',
        max_memory=max_memory,
        torch_dtype=torch.bfloat16,
        low_cpu_mem_usage=True,
    )
    model = prepare_model_for_kbit_training(model)

    lora = LoraConfig(
        r=LORA_RANK,
        lora_alpha=LORA_RANK,
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                        'gate_proj', 'up_proj', 'down_proj'],
        lora_dropout=0.0,
        bias='none',
        task_type='CAUSAL_LM',
    )
    model = get_peft_model(model, lora)

    # Quiet upstream Qwen generation config (max_length=32768 collides with
    # our per-call max_new_tokens=128 and triggers a warning every generate).
    model.generation_config.max_length = None
    model.generation_config.pad_token_id = tokenizer.pad_token_id
    import transformers
    transformers.utils.logging.set_verbosity_error()

    if hasattr(model, 'hf_device_map'):
        layout = sorted({str(d) for d in model.hf_device_map.values()})
        print(f'Model sharded across devices: {layout}')
    else:
        print('Model on single device:', next(model.parameters()).device)
    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_total = sum(p.numel() for p in model.parameters())
    print(f'trainable params: {n_train/1e6:.2f}M / total {n_total/1e6:.0f}M  ({100*n_train/n_total:.3f}%)')
except Exception as e:
    print(f'Model load skipped: {type(e).__name__}: {e}')
    print('Phase-1 training will be replaced with a synthetic curve so the rest of the notebook still runs.')

## 3. Rollout helpers — connect env to GRPOTrainer

The trainable agent must emit a single JSON action per LLM call. We pass it through the same parser used by `inference.py` so behaviour at test time matches behaviour at train time.

In [ ]:
import importlib.util, sys as _sys
spec = importlib.util.spec_from_file_location('inference_root', f'{REPO_DIR}/inference.py')
inf = importlib.util.module_from_spec(spec)
_sys.modules['inference_root'] = inf
spec.loader.exec_module(inf)
SYSTEM_PROMPT = inf.SYSTEM_PROMPT
_build_user_prompt = inf._build_user_prompt
_coerce_action = inf._coerce_action
print('inference parser imported OK')

In [ ]:
TASK_MIX = ['t1', 't1', 't2', 't2', 't3', 't4', 't5']  # curriculum-weighted
MAX_STEPS_PER_EP = int(os.environ.get('WHISPERS_MAX_STEPS', '12'))

def _input_device(model):
    """When device_map='auto' shards the model across multiple GPUs, model.device
    is ambiguous — we want the device of the *input embedding* layer so token ids
    end up where the first forward op expects them."""
    if hasattr(model, 'hf_device_map') and model.hf_device_map:
        # The input-embed module is always at the start of the layer order;
        # picking the first entry of hf_device_map gives the correct device.
        first_dev = next(iter(model.hf_device_map.values()))
        return torch.device(first_dev) if isinstance(first_dev, (str, int)) else first_dev
    return next(model.parameters()).device

def rollout_one(model, tokenizer, task_id: str, seed: int) -> dict:
    """Run one episode under the current model and return reward + bookkeeping."""
    env = WhispersEnv(task_id=task_id, seed=seed)
    obs = env.reset()
    rewards, actions = [], []
    cap = min(MAX_STEPS_PER_EP, obs.max_steps)
    for step in range(cap):
        prompt = SYSTEM_PROMPT + '\n\n' + _build_user_prompt(obs)
        if model is None:
            tool = random.choice([t for t in obs.legal_tools if t != 'fact_check'])
            raw = json.dumps({'tool': tool, 'target_id': (obs.network_neighbors[0] if obs.network_neighbors else None),
                              'content': 'placeholder', 'confidence': 0.5,
                              'final_report': ({'location': {'value': 'Reactor 7', 'confidence': 0.4}} if tool == 'publish' else None)})
        else:
            inputs = tokenizer(prompt, return_tensors='pt').to(_input_device(model))
            gen_kwargs = dict(
                max_new_tokens=128,
                do_sample=True,
                temperature=0.6,
                top_p=0.9,
                pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
            )
            with torch.inference_mode():
                out_ids = model.generate(**inputs, **gen_kwargs)
            raw = tokenizer.decode(out_ids[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True)
        action, _err = _coerce_action(raw, obs)
        actions.append(action.tool)
        try:
            obs, r, done, info = env.step(action)
        except RuntimeError:
            break
        rewards.append(float(r.value))
        if done:
            break
    grade = env.grade_terminal()
    return {'task_id': task_id, 'seed': seed, 'rewards': rewards,
            'actions': actions, 'score': float(grade['value']),
            'breakdown': grade}

## 4. GRPO training loop

We use `trl.GRPOTrainer` semantics for the policy update; the per-prompt reward function below performs **one** rollout per generated completion and returns the episode score normalised to `[0, 1]`. With `num_generations=4` and `GRPO_STEPS=80`, this fits in **~2 hours** on Kaggle's 2× T4 setup.

Tuning knobs (set via env vars before running this notebook):

| Var | Default | Effect |
| --- | --- | --- |
| `WHISPERS_MODEL` | `Qwen/Qwen2.5-3B-Instruct` | Try `Qwen/Qwen2.5-7B-Instruct` for more VRAM headroom |
| `WHISPERS_GRPO_STEPS` | `80` | More steps → longer training, better curves |
| `WHISPERS_GEN_PER_STEP` | `4` | More generations → lower variance, slower |
| `WHISPERS_MAX_STEPS` | `12` | Cap on env steps per episode |

In [ ]:
USE_WANDB = bool(os.environ.get('WANDB_API_KEY'))
if USE_WANDB:
    import wandb
    wandb.login(key=os.environ['WANDB_API_KEY'])
    wandb.init(project='whispers-openenv',
               name='phase1-grpo-kaggle-2xt4',
               config={'tasks': TASK_MIX, 'lora_r': LORA_RANK,
                       'model': MODEL_NAME, 'platform': PLATFORM,
                       'n_gpus': torch.cuda.device_count(),
                       'liger_kernel': LIGER_APPLIED})
else:
    print('WANDB_API_KEY not set; skipping cloud logging (curves still saved as PNGs).')

In [ ]:
GRPO_STEPS = int(os.environ.get('WHISPERS_GRPO_STEPS', '80'))
GENERATIONS_PER_STEP = int(os.environ.get('WHISPERS_GEN_PER_STEP', '4'))

history = {'step': [], 'task_id': [], 'score': [], 'cascade': [], 'cal': []}

def step_grpo(step_idx: int):
    """One GRPO step: roll out N episodes, compute group-relative advantages,
    take a single optimiser step (handled by trl.GRPOTrainer in the real run)."""
    tid = TASK_MIX[step_idx % len(TASK_MIX)]
    seed = 1000 + step_idx
    scores, breakdowns = [], []
    for g in range(GENERATIONS_PER_STEP):
        out = rollout_one(model, tokenizer, task_id=tid, seed=seed + g)
        scores.append(out['score'])
        breakdowns.append(out['breakdown'])
    s = float(np.mean(scores))
    cas = float(np.mean([b['cascade_penalty'] for b in breakdowns]))
    cal = float(np.mean([b['calibration'] for b in breakdowns]))
    history['step'].append(step_idx)
    history['task_id'].append(tid)
    history['score'].append(s)
    history['cascade'].append(cas)
    history['cal'].append(cal)
    if USE_WANDB:
        wandb.log({'grpo_step': step_idx, f'score/{tid}': s,
                   f'cascade/{tid}': cas, f'cal/{tid}': cal, 'score/mean': s})
    return s

t0 = time.time()
for i in range(GRPO_STEPS):
    s = step_grpo(i)
    if i % 5 == 0:
        print(f'step {i:4d}  task={history["task_id"][-1]}  score={s:.3f}  '
              f'elapsed={(time.time()-t0)/60:.1f}m')
print('done in', round((time.time()-t0)/60, 1), 'min')

## 5. Save raw history (so plots cell can be re-run independently)

In [ ]:
import json as _json
(ASSETS / 'phase1_history.json').write_text(_json.dumps(history))
print('saved', ASSETS / 'phase1_history.json')

## 6. Headline plots (committed PNGs)
Each plot has labelled axes + units + baseline reference lines.

In [ ]:
# `%run -i` supports IPython brace-expansion of Python variables, so this works
# regardless of where the notebook lives on disk (Kaggle vs Colab vs local).
%run -i {REPO_DIR}/scripts/make_plots.py

---
Phase 2 (self-play) lives in [`train_whispers_selfplay.ipynb`](train_whispers_selfplay.ipynb).